# Pertemuan 2 — Rekursi dan Persamaan Rekurens
Master Theorem, recursion tree, substitusi, dan Akra-Bazzi (pengantar) — diverifikasi secara numerik.

In [ ]:
import math
from functools import lru_cache
import numpy as np
import matplotlib.pyplot as plt

## 1. Solver Master Theorem
Untuk T(n) = a·T(n/b) + Θ(nᵈ):  bandingkan d dengan log_b(a).

In [ ]:
def master(a, b, d):
    crit = math.log(a, b)
    if abs(d - crit) < 1e-9:  return f"Θ(n^{d:g} · log n)    [kasus 2, log_b a = {crit:.3f}]"
    if d < crit:              return f"Θ(n^{crit:.3f})        [kasus 1: daun mendominasi]"
    return f"Θ(n^{d:g})            [kasus 3: akar mendominasi]"

for a, b, d, name in [(2,2,1,"Merge Sort"), (1,2,0,"Binary Search"), (7,2,2,"Strassen"),
                      (8,2,2,"Matriks D&C naif"), (3,2,1,"Karatsuba"), (2,2,2,"contoh kasus 3")]:
    print(f"T(n)={a}T(n/{b})+n^{d}  {name:18s} → {master(a, b, d)}")

## 2. Recursion tree: biaya per level
T(n)=2T(n/2)+n → tiap level berbiaya n, ada log₂n level.

In [ ]:
n = 1024
print("level  #subsoal  ukuran  biaya/level")
for lvl in range(int(math.log2(n)) + 1):
    cnt, size = 2**lvl, n // 2**lvl
    print(f"{lvl:>5} {cnt:>9} {size:>7} {cnt*size:>11}")
print("Total ≈ n·log2(n) + n =", n * int(math.log2(n)) + n)

## 3. Verifikasi numerik: hitung T(n) lalu bandingkan dengan tebakan
Substitusi: tebak T(n) ≤ c·n log n lalu cek rasio T(n)/(n log₂ n) konvergen ke konstanta.

In [ ]:
@lru_cache(None)
def T_merge(n): return 1 if n <= 1 else 2 * T_merge(n // 2) + n
@lru_cache(None)
def T_bin(n):   return 1 if n <= 1 else T_bin(n // 2) + 1
@lru_cache(None)
def T_stra(n):  return 1 if n <= 1 else 7 * T_stra(n // 2) + n * n

for k in [4, 8, 12, 16, 20]:
    n = 2**k
    print(f"n=2^{k:<2} merge/(n log n)={T_merge(n)/(n*k):.3f}  binsearch/log n={T_bin(n)/k:.3f}  strassen/n^2.807={T_stra(n)/n**math.log2(7):.3f}")

## 4. Pembuktian empiris: Merge Sort memang melakukan ≈ n log₂ n perbandingan

In [ ]:
import random
def merge_sort_count(a):
    if len(a) <= 1: return a, 0
    m = len(a) // 2
    L, cl = merge_sort_count(a[:m]); R, cr = merge_sort_count(a[m:])
    out, i, j, c = [], 0, 0, 0
    while i < len(L) and j < len(R):
        c += 1
        if L[i] <= R[j]: out.append(L[i]); i += 1
        else: out.append(R[j]); j += 1
    out += L[i:] + R[j:]
    return out, cl + cr + c

ns = [2**k for k in range(6, 15)]
cnt = [merge_sort_count(random.sample(range(10*n), n))[1] for n in ns]
plt.plot(ns, cnt, "o-", label="terukur"); plt.plot(ns, [n*math.log2(n) for n in ns], "--", label="n·log₂n")
plt.xscale("log"); plt.yscale("log"); plt.legend(); plt.xlabel("n"); plt.ylabel("perbandingan"); plt.show()

## 5. Di luar Master Theorem: Akra–Bazzi (pengantar)
T(n) = T(n/2) + T(n/3) + n. Cari p dengan (1/2)ᵖ + (1/3)ᵖ = 1. Karena p < 1 dan f(n)=n, maka T(n)=Θ(n).

In [ ]:
def solve_p(ws, lo=0.0, hi=5.0):
    for _ in range(100):
        mid = (lo + hi) / 2
        if sum(w**mid for w in ws) > 1: lo = mid
        else: hi = mid
    return mid
p = solve_p([1/2, 1/3]); print("p =", round(p, 4))
@lru_cache(None)
def T_ab(n): return 1 if n <= 1 else T_ab(n // 2) + T_ab(n // 3) + n
for n in [10**3, 10**5, 10**7]:
    print(n, "T(n)/n =", round(T_ab(n) / n, 3))

## 6. Latihan
1. Selesaikan dengan `master()` lalu verifikasi numerik: T(n)=4T(n/2)+n, T(n)=4T(n/2)+n², T(n)=4T(n/2)+n³.
2. Tulis rekurens untuk *binary search* dan *quickselect* (kasus rata-rata) dan buktikan dengan substitusi.
3. Tentukan Θ untuk T(n)=T(n/2)+T(n/4)+T(n/8)+n dengan Akra–Bazzi.

In [ ]:
def T_latihan(n):
    pass  # TODO